In [4]:
import polars as pl
import datetime


In [5]:
df = (
    pl.read_csv("../data/ishares.csv", separator=";")
    .with_columns(incept_date=pl.col("Incept. Date").str.to_date("%b %d,%Y"))
    .with_columns(
        net_assets=pl.col("Net Assets (USD)")
        .str.replace_all(".", "", literal=True)
        .str.replace(",", ".", literal=True)
        .cast(pl.Float64)
    )
    .drop("Incept. Date", "Net Assets (USD)")
    .with_columns(days_since_inception=(pl.lit(datetime.date.today()) - pl.col("incept_date")).dt.total_days())
    .filter(pl.col("days_since_inception") > 2000)
    .filter(pl.col("Investment Style") == "Index")
    .sort("net_assets", descending=True)
    .group_by("Asset Class", "Sub Asset Class", "Region", "Market")
    .first()
)

df

Asset Class,Sub Asset Class,Region,Market,Ticker,Name,ISIN,Location,Investment Style,incept_date,net_assets,days_since_inception
str,str,str,str,str,str,str,str,str,date,f64,i64
"""Commodity""","""Multi Commodity""","""Global""","""Developed""","""GSG""","""iShares S&P GSCI Commodity-Ind…","""US46428R1077""","""Broad""","""Index""",2006-07-10,1.0364e9,6989
"""Equity""","""Large/Mid Cap""","""Global""","""Developed""","""EFA""","""iShares MSCI EAFE ETF""","""US4642874659""","""Broad""","""Index""",2001-08-14,6.6031e10,8780
"""Fixed Income""","""High Yield""","""Global""","""Developed""","""GHYG""","""iShares US & Intl High Yield C…","""US4642861789""","""Broad""","""Index""",2012-04-03,1.7554e8,4895
"""Fixed Income""","""Municipals""","""North America""","""Developed""","""MUB""","""iShares National Muni Bond ETF""","""US4642884146""","""United States""","""Index""",2007-09-07,3.8551e10,6565
"""Fixed Income""","""High Yield""","""Global""","""Emerging""","""EMHY""","""iShares J.P. Morgan EM High Yi…","""US4642862852""","""Broad""","""Index""",2012-04-03,4.8940e8,4895
…,…,…,…,…,…,…,…,…,…,…,…
"""Equity""","""Large/Mid Cap""","""North America""","""Developed""","""IWF""","""iShares Russell 1000 Growth ET…","""US4642876142""","""United States""","""Index""",2000-05-22,1.1690e11,9229
"""Equity""","""Small Cap""","""North America""","""Developed""","""IJR""","""iShares Core S&P Small-Cap ETF""","""US4642878049""","""United States""","""Index""",2000-05-22,8.5654e10,9229
"""Equity""","""Large/Mid Cap""","""Asia Pacific""","""Developed""","""EWJ""","""iShares MSCI Japan ETF""","""US46434G8226""","""Japan""","""Index""",1996-03-12,1.5386e10,10761


In [ ]:
df.write_json("etfs.json")

In [12]:
# Add instruments
import requests

requests.post(
    url="http://localhost:8000/api/instruments",
    json=[dict(name=item["Name"], ticker=item["Ticker"], currency="USD") for item in df.rows(named=True)],
)

<Response [200]>

Asset Class,Sub Asset Class,Region,Market,Ticker,Name,ISIN,Location,Investment Style,incept_date,net_assets,days_since_inception
str,str,str,str,str,str,str,str,str,date,f64,i64
"""Fixed Income""","""Corporates""","""North America""","""Developed""","""LDRC""","""iShares® iBonds® 1-5 Year Corp…","""US46438G5392""","""United States""","""Index""",2024-11-07,9115786.9,294
"""Digital Assets""","""Cryptoassets""","""North America""","""Developed""","""IBIT""","""iShares Bitcoin Trust ETF""","""US46438F1012""","""United States""","""Index""",2024-01-05,8.3491e10,601
"""Fixed Income""","""Corporates""","""Global""","""Emerging""","""BEMB""","""iShares J.P. Morgan Broad USD …","""US46436E2625""","""Broad""","""Index""",2023-02-22,4.2696e7,918
"""Equity""","""All Cap""","""Kuwait""","""Emerging""","""KWT""","""iShares MSCI Kuwait ETF""","""US46436E8176""","""Kuwait""","""Index""",2020-09-01,8.3148e7,1822
"""Multi Asset""","""Multi Strategy""","""North America""","""Developed""","""EAOA""","""iShares ESG Aware 80/20 Aggres…","""US46436E6683""","""United States""","""Index""",2020-06-12,3.2664e7,1903
…,…,…,…,…,…,…,…,…,…,…,…
"""Equity""","""Large/Mid Cap""","""North America""","""Developed""","""IWF""","""iShares Russell 1000 Growth ET…","""US4642876142""","""United States""","""Index""",2000-05-22,1.1690e11,9229
"""Equity""","""Small Cap""","""North America""","""Developed""","""IJR""","""iShares Core S&P Small-Cap ETF""","""US4642878049""","""United States""","""Index""",2000-05-22,8.5654e10,9229
"""Equity""","""Large Cap""","""North America""","""Developed""","""IVV""","""iShares Core S&P 500 ETF""","""US4642872000""","""United States""","""Index""",2000-05-15,6.6321e11,9236


In [15]:
df.to_dicts()

[{'Asset Class': 'Fixed Income',
  'Sub Asset Class': 'Multi Sectors',
  'Region': 'North America',
  'Market': 'Developed',
  'Ticker': 'AGG',
  'Name': 'iShares Core U.S. Aggregate Bond ETF',
  'ISIN': 'US4642872265',
  'Location': 'United States',
  'Investment Style': 'Index',
  'incept_date': datetime.date(2003, 9, 22),
  'net_assets': 130597886050.06},
 {'Asset Class': 'Digital Assets',
  'Sub Asset Class': 'Cryptoassets',
  'Region': 'North America',
  'Market': 'Developed',
  'Ticker': 'IBIT',
  'Name': 'iShares Bitcoin Trust ETF',
  'ISIN': 'US46438F1012',
  'Location': 'United States',
  'Investment Style': 'Index',
  'incept_date': datetime.date(2024, 1, 5),
  'net_assets': 83491306821.88},
 {'Asset Class': 'Equity',
  'Sub Asset Class': 'Large/Mid Cap',
  'Region': 'Asia Pacific',
  'Market': 'Emerging',
  'Ticker': 'INDA',
  'Name': 'iShares MSCI India ETF',
  'ISIN': 'US46429B5984',
  'Location': 'India',
  'Investment Style': 'Index',
  'incept_date': datetime.date(2012,